## Membaca dataset dab menyiapkan spark session

In [3]:
import spark_session
import pandas as pd

app = spark_session.SparkApp("Tugas12-pipelineMl")
spark = app.getSession()

data = pd.read_csv("file:///home/asfadani/praktikum-bigdata/Praktikum-BigData/tugas12_ulasan.csv")
df = spark.createDataFrame(data)
df.show(5)

Memulai SparkSession untuk Tugas12-pipelineMl
SparkSession siap. Versi Spark: 3.5.9


+--------------+----------+-----------+---+------------------+--------------+
|     nama_menu|    cabang|waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|
+--------------+----------+-----------+---+------------------+--------------+
|     Kopi Susu|Yogyakarta|       Sore|  5|                 5|             1|
|        Es Teh|Yogyakarta|       Pagi|  1|                21|             0|
|    Cappuccino|Yogyakarta|       Sore|  5|                15|             0|
|Kentang Goreng|  Semarang|      Malam|  2|                 5|             1|
|     Americano|Yogyakarta|      Malam|  2|                10|             0|
+--------------+----------+-----------+---+------------------+--------------+
only showing top 5 rows



## A. Membangun ML Pipeline (bobot 25%)

Buat Pipeline dengan tahapan: StringIndexer untuk nama_menu, cabang, waktu_pesan → OneHotEncoder untuk ketiganya → VectorAssembler (gabungkan hasil one-hot encoding dengan qty dan waktu_tunggu_menit) → LogisticRegression dengan labelCol="ulasan_positif".

In [8]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline

# Indexing
idx_menu = StringIndexer(inputCol="nama_menu", outputCol="menu_idx")
idx_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
idx_pesan = StringIndexer(inputCol="waktu_pesan", outputCol="pesan_idx")

# OHE
ohe = OneHotEncoder(
    inputCols=["menu_idx", "cabang_idx", "pesan_idx"],
    outputCols=["menu_vec", "cabang_vec", "pesan_vec"]
)

# Vector
assembler = VectorAssembler(
    inputCols=["menu_vec", "cabang_vec", "pesan_vec", "qty", "waktu_tunggu_menit"],
    outputCol="features"
)

# model lr
lr = LogisticRegression(featuresCol="features", labelCol="ulasan_positif")

# gabung jadi pipeline
pipeline = Pipeline(stages=[idx_menu, idx_cabang, idx_pesan, ohe, assembler, lr])

print("tahap didefinisikan dan pipeline disusun!")

tahap didefinisikan dan pipeline disusun!


## B. Pelatihan & Evaluasi (bobot 25%)

Bagi data 80:20 (seed=77), latih pipeline, lalu evaluasi pada data testing menggunakan accuracy, F1-score, dan AUC-ROC. Tampilkan juga confusion matrix sederhana menggunakan
groupBy.

In [16]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

# Split data
train, test = df.randomSplit([0.8, 0.2], seed=77)

# train model
model = pipeline.fit(train)

# uji ke data test
predict = pipeline_model_columns = model.transform(test)

predict.select("nama_menu", "cabang", "waktu_pesan", "qty", "waktu_tunggu_menit", "ulasan_positif", "prediction", "probability").show(5, truncate=False)

# evaluasi model
ev_acc = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="accuracy")
accuracy = ev_acc.evaluate(predict)

ev_f1 = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="f1")
f1 = ev_f1.evaluate(predict)

ev_auc = BinaryClassificationEvaluator(labelCol="ulasan_positif", metricName="areaUnderROC")
auc = ev_auc.evaluate(predict)

print(f"Accuracy : {accuracy:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"AUC-ROC  : {auc:.4f}")

predict.groupBy("ulasan_positif", "prediction").count().orderBy("ulasan_positif", "prediction").show()

+---------+----------+-----------+---+------------------+--------------+----------+----------------------------------------+
|nama_menu|cabang    |waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|prediction|probability                             |
+---------+----------+-----------+---+------------------+--------------+----------+----------------------------------------+
|Americano|Magelang  |Pagi       |2  |7                 |1             |1.0       |[0.2746339940974035,0.7253660059025965] |
|Americano|Magelang  |Siang      |1  |6                 |1             |1.0       |[0.2349787472600326,0.7650212527399674] |
|Americano|Semarang  |Malam      |3  |24                |0             |0.0       |[0.9416348083845739,0.05836519161542608]|
|Americano|Semarang  |Pagi       |5  |13                |0             |0.0       |[0.6081956094595845,0.3918043905404155] |
|Americano|Yogyakarta|Malam      |1  |18                |1             |0.0       |[0.7708330905932524,0.22916690940674755]|


## C. Model Pembanding (bobot 20%)

Latih RandomForestClassifier (numTrees=30) menggunakan pipeline dengan lima tahap feature engineering yang identik dari bagian A (hanya tahap terakhir yang diganti). Bandingkan ketiga metrik evaluasinya dengan model Logistic Regression pada bagian B.  

In [19]:
from pyspark.ml.classification import RandomForestClassifier

# Model rf
rf = RandomForestClassifier(featuresCol='features', labelCol="ulasan_positif", numTrees=30)

# pipeline rf
pipeline_rf = Pipeline(stages=[idx_menu, idx_cabang, idx_pesan, ohe, assembler, rf])

model_rf = pipeline_rf.fit(train)
predict_rf = model_rf.transform(test)

acc_rf = ev_acc.evaluate(predict_rf)
auc_rf = ev_auc.evaluate(predict_rf)

# evaluasi model
print("=== Perbandingan Hasil ===")
print(f"Logistic Regression -> Accuracy: {accuracy:.4f} | AUC: {auc:.4f}")
print(f"Random Forest        -> Accuracy: {acc_rf:.4f} | AUC: {auc_rf:.4f}")

=== Perbandingan Hasil ===
Logistic Regression -> Accuracy: 0.7677 | AUC: 0.8345
Random Forest        -> Accuracy: 0.7661 | AUC: 0.8131


## D. Analisis Kesalahan Model (bobot 15%)

Dari hasil prediksi model terbaik (pilih salah satu dari B atau C berdasarkan AUC tertinggi), tampilkan 10 baris di mana model salah memprediksi (label != prediction). Amati: apakah ada pola tertentu pada kesalahan tersebut (mis. mayoritas terjadi pada waktu_tunggu_menit yang berada di sekitar ambang batas tertentu)?

In [31]:
from pyspark.sql.functions import col

predict.select("nama_menu", "cabang", "waktu_pesan", "qty", "waktu_tunggu_menit", "ulasan_positif", "prediction").filter(col("ulasan_positif") != col("prediction")).show(10)

+--------------+----------+-----------+---+------------------+--------------+----------+
|     nama_menu|    cabang|waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|prediction|
+--------------+----------+-----------+---+------------------+--------------+----------+
|     Americano|Yogyakarta|      Malam|  1|                18|             1|       0.0|
|     Americano|Yogyakarta|      Malam|  2|                10|             0|       1.0|
|    Cappuccino|  Magelang|      Malam|  2|                12|             0|       1.0|
|    Cappuccino|Yogyakarta|       Sore|  3|                13|             1|       0.0|
|    Cappuccino|Yogyakarta|       Sore|  3|                16|             1|       0.0|
|Kentang Goreng|  Magelang|      Siang|  2|                 9|             0|       1.0|
|Kentang Goreng|  Semarang|       Sore|  5|                12|             1|       0.0|
|     Kopi Susu|Yogyakarta|      Siang|  3|                11|             0|       1.0|
|     Kopi Susu|Yogya

>Jika dilihat dari hasil prediksi, terdapat salah satu pola dimana model salah menebak, yaitu pada waktu ulasan. Model memprediksi dari hasil mempelajari aturan sederhana dimana model akan menebak ulasan negatif(0) jika waktu tunggu >= 12 atau positif jika waktu tunggu <=12. Untuk leb

In [30]:
from pyspark.sql.functions import count, avg, min as Min, max as Max

# filter hanya baris yang salah prediksi
salah_prediksi = predict.filter(col("ulasan_positif") != col("prediction"))

salah_prediksi.groupBy("ulasan_positif", "prediction") \
    .agg(
        count("*").alias("jumlah_salah"),
        Min("waktu_tunggu_menit").alias("min_tunggu"),
        avg("waktu_tunggu_menit").alias("rata2_tunggu"),
        Max("waktu_tunggu_menit").alias("max_tunggu")
    ).show()

+--------------+----------+------------+----------+------------------+----------+
|ulasan_positif|prediction|jumlah_salah|min_tunggu|      rata2_tunggu|max_tunggu|
+--------------+----------+------------+----------+------------------+----------+
|             1|       0.0|          81|        11|16.333333333333332|        24|
|             0|       1.0|          63|         2| 8.444444444444445|        12|
+--------------+----------+------------+----------+------------------+----------+



>Dapat dilihat bahwa memang benar model belajar dari aturan sederhana jika <=12 model akan menebak positif dan jika >= 12 model menebak negatif.

## E. Kesimpulan Bisnis (bobot 15%)

Tulis pada markdown cell (minimal 100 kata): berdasarkan model yang telah dibangun, apa rekomendasi konkret yang dapat diberikan kepada manajemen "Kopi Nusantara" untuk meningkatkan kepuasan pelanggan? Kaitkan dengan fitur yang paling berpengaruh terhadap prediksi (boleh merujuk pada pola yang ditemukan di bagian D).

>Berdasarkan hasil pengamatan terhadap data ulasan pelanggan Kopi Nusantara, faktor yang paling menentukan apakah pelanggan memberikan ulasan positif atau negatif adalah lama waktu tunggu pesanan. Pelanggan cenderung merasa puas dan memberikan ulasan positif apabila pesanan selesai dalam waktu maksimal 12 menit, sedangkan waktu tunggu di atas 12 menit sangat berisiko memicu keluhan dan ulasan negatif.
Oleh karena itu, manajemen Kopi Nusantara disarankan untuk menerapkan standar batas waktu penyajian maksimal 10–12 menit per pesanan sebagai target utama operasional di seluruh cabang. Pada jam jam sibuk seperti sore dan malam hari, manajemen perlu menambah tenaga barista atau menyiapkan bahan setengah jadi untuk menu populer agar antrean tidak menumpuk. Selain itu, kasir sebaiknya memberikan informasi perkiraan waktu tunggu sejak awal pemesanan jika antrean sedang padat, serta memantau kinerja cabang yang sering melewati batas 12 menit agarkepuasan pelanggan tetap terjaga.